In [1]:
### Analysis of PFC data from Mante et al 2013

In [2]:
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d
import torch
from scipy import io as spio
from sklearn.decomposition import PCA

def loadmat(filename):
    '''
    this function should be called instead of direct spio.loadmat
    as it cures the problem of not properly recovering python dictionaries
    from mat files. It calls the function check keys to cure all entries
    which are still mat-objects

    from: `StackOverflow <http://stackoverflow.com/questions/7008608/scipy-io-loadmat-nested-structures-i-e-dictionaries>`_
    '''
    data = spio.loadmat(filename, struct_as_record=False, squeeze_me=True)
    return _check_keys(data)


def _check_keys(dict):
    '''
    checks if entries in dictionary are mat-objects. If yes
    todict is called to change them to nested dictionaries
    '''
    for key in dict:
        if isinstance(dict[key], spio.matlab.mio5_params.mat_struct):
            dict[key] = _todict(dict[key])
    return dict


def _todict(matobj):
    '''
    A recursive function which constructs from matobjects nested dictionaries
    '''
    dict = {}
    for strg in matobj._fieldnames:
        elem = matobj.__dict__[strg]
        if isinstance(elem, spio.matlab.mio5_params.mat_struct):
            dict[strg] = _todict(elem)
        else:
            dict[strg] = elem
    return dict




def contrast_means(df):
    df.stim_dir = df.stim_dir.astype(float)
    df.stim_col2dir = df.stim_col2dir.astype(float)
    dir_means = np.mean(np.stack(df.groupby('unit_id')['stim_dir'].apply(np.unique).reset_index(
        name='unique_contrasts').unique_contrasts.values), axis=0)
    col2dir_means = np.mean(np.stack(df.groupby('unit_id')['stim_col2dir'].apply(np.unique).reset_index(
        name='unique_contrasts').unique_contrasts.values), axis=0)
    return np.round(dir_means, 2), np.round(col2dir_means, 2)


def replace_dir(x, dir_means):
    """
    Replace each contrast
    """
    unique_dir = np.unique(x)

    def fn(dir):
        return dir_means[np.argwhere(unique_dir == dir)].item()

    return list(map(fn, x))


def replace_col2dir(x, col2dir_means):
    """
    Replace each contrast
    """
    unique_col2dir = np.unique(x)

    def fn(col2dir):
        return col2dir_means[np.argwhere(unique_col2dir == col2dir)].item()

    return list(map(fn, x))





matdata = loadmat('Data/Mante/Monkey_' + animal + '/dataT.mat')

# Reorganize into pandas frame.
rows = []
nun = len(matdata['dataT']['unit'])
for unit in range(nun):
    n_trials = matdata['dataT']['unit'][unit].response.shape[0]
    for k in range(n_trials):
        # if matdata['dataT']['unit'][unit].task_variable.correct[k]==1:
        rows.append({'unit_id': unit,
                     'trial': k,
                     'stim_dir': matdata['dataT']['unit'][unit].task_variable.stim_dir[k],
                     'stim_col2dir': matdata['dataT']['unit'][unit].task_variable.stim_col2dir[k],
                     'context': matdata['dataT']['unit'][unit].task_variable.context[k],
                     'choice': matdata['dataT']['unit'][unit].task_variable.targ_dir[k],
                     'correct': matdata['dataT']['unit'][unit].task_variable.correct[k],
                     'response': matdata['dataT']['unit'][unit].response[k]})
df = pd.DataFrame(rows)

# Rename context
dct = {1: 'motion', -1: 'color'}
df.context = [*map(dct.get, df.context.values)]

# Group contrasts
dir_means, col2dir_means = contrast_means(df)
df.stim_dir = df.groupby('unit_id')['stim_dir'].transform(lambda x: replace_dir(x, dir_means))
df.stim_col2dir = df.groupby('unit_id')['stim_col2dir'].transform(lambda x: replace_col2dir(x, col2dir_means))

# Get units that have at least 10 trials for all correct conditions
df_correct = df[df.correct == 1]
df_correct["Condition"] = \
    df_correct.groupby(['stim_dir', 'stim_col2dir', 'context', 'choice']).grouper.group_info[0]
counts = df_correct.groupby(['unit_id', 'Condition']).size().reset_index(name="counts")
min_counts = counts.groupby('unit_id').counts.min().reset_index()
good_units = min_counts[min_counts.counts >= 4]['unit_id'].unique()
df = df[df.unit_id.isin(good_units)]










In [3]:
n_boot = 25
p  = .8
rows = []
for unit_id in df.unit_id.unique():
    unit_df = df[df.unit_id==unit_id]
    for condition in unit_df.Condition.unique():
        for bootstrap in range(n_boot):
            sample_df = (unit_df[unit_df.Condition == condition]).sample( frac=p, replace=True)
            rows.append({'unit_id': unit_id, 
                         'bootstrap': bootstrap,
                         'condition':condition,
                         'stim_dir': sample_df.stim_dir.unique().item(),
                         'stim_col2dir': sample_df.stim_col2dir.unique().item(),
                         'context': sample_df.context.unique().item(),
                         'choice': sample_df.choice.unique().item(),
                         'correct':sample_df.correct.unique().item(),
                         'response':np.mean(np.stack(sample_df.response.values,axis=0),axis=0)})

In [4]:
new_df = pd.DataFrame(rows)
new_df.to_pickle("pfc_ar.pkl") 

## Load PFC data

In [42]:
df = pd.read_pickle("pfc_ar.pkl")  

In [34]:
# Smooth responses
sigma = 1
df['response'] = df.response.apply(gaussian_filter1d, args=[sigma])

In [27]:
# Z-score
df['center'] = df.groupby(['unit_id']).response.transform(lambda x: np.mean(np.stack(x)))
df['std'] = df.groupby(['unit_id']).response.transform(lambda x: np.std(np.stack(x)))
df['response'] = (df['response'] - df['center']) / df['std']

In [35]:
# Restrict to intersection of conditions across units
condition_sets = df.groupby('unit_id').condition.apply(lambda x: set(np.stack(x.values))).reset_index()
u = set.intersection(*list(condition_sets.condition.values))
df = df[df.condition.isin(u)]
df = df[df.correct==1]

In [36]:
train = df.groupby(['bootstrap','stim_dir', 'stim_col2dir', 'context', 'choice', 'correct'])['response'].apply(
        lambda x: np.stack(x)).reset_index()

In [29]:
# Remove condition independent responses
x_t = np.mean(np.stack(train.response.values), axis=0)
train.response = train.response.apply(lambda x: x - x_t)

In [31]:
# PCA
n_components = .5
from sklearn.decomposition import PCA
pca = PCA(n_components=n_components, svd_solver='full')
N = train.response.values[0].shape[1]
pca.fit(np.reshape(np.stack(train.response.values, axis=0), (-1, N)))
train.response = train.response.apply(lambda x: pca.inverse_transform(pca.transform(x)))

In [37]:
### Reshape data into 3d array
n_boot = 25
k = len(df.condition.unique()) 
t = len(df.response.values[0])
n = len(df.unit_id.unique())
unit_ids = df.unit_id.unique()
conditions = list(u)

flattened_conditions = []
y = torch.zeros(n_boot * k, t, n)
trial = 0
for index, row in train.iterrows():

   # condition = np.argwhere(conditions==row['condition'])
   # unit = np.argwhere(unit_ids==row['unit_id'])
    y[trial,:,:] = torch.tensor(row['response']).float().t()
    trial+=1
    flattened_conditions.append([row['context'],
                                 row['stim_dir'],
                                 row['stim_col2dir']])

In [55]:
y.shape

torch.Size([1800, 15, 483])

## Construct inputs and targets for latent net

In [ ]:
trial_events = {'n_t': 150,
                'cue_on': 0,
                'cue_off': 150,
                'stim_on': 75,
                'stim_off': 150,
                'dec_on': 125,
                'dec_off': 150,
                'data_on': 75,
                'qui_on': 0,
                'qui_off': 0}


z_mask = np.hstack((np.arange(trial_events['qui_on'], trial_events['qui_off']),
                    np.arange(trial_events['dec_on'], trial_events['n_t'])))

In [43]:

def generate_input_target_stream(context, motion_coh, color_coh, alpha, baseline, sigma_in, n_t, cue_on, cue_off,
                                 stim_on, stim_off, dec_on, dec_off, data_on, qui_on, qui_off):
    """
    Generate input and target sequence for a given set of trial conditions.

    :param t:
    :param tau:
    :param cue:
    :param motion_coh:
    :param color_coh:
    :param baseline:
    :param alpha:
    :param sigma_in:
    :param cue_on:
    :param cue_off:
    :param stim_on:
    :param stim_off:
    :param dec_off:
    :param dec_on:

    :return: input stream
    :return: target stream

    """
    # Convert trial events to discrete time
    # cue_off=n_t
    # stim_off=n_t
    #  dec_off=n_t
    #
    n_in = 6

    # Transform coherence to signal
    motion_r = (1 + motion_coh) / 2
    motion_l = 1 - motion_r
    color_r = (1 + color_coh) / 2
    color_l = 1 - color_r

    # Cue input stream
    cue_input = np.zeros([n_t, n_in])
    if context == "motion":
        cue_input[cue_on:cue_off, 0] = 1.2 * np.ones(
            [cue_off - cue_on, 1]).squeeze()
    else:
        cue_input[cue_on:cue_off, 1] = 1.2 * np.ones(
            [cue_off - cue_on, 1]).squeeze()

    # Motion input stream
    motion_input = np.zeros([n_t, n_in])
    motion_input[stim_on:stim_off, 2] = motion_r * np.ones([stim_off - stim_on])
    motion_input[stim_on:stim_off, 3] = motion_l * np.ones([stim_off - stim_on])

    # Color input stream
    color_input = np.zeros([n_t, n_in])
    color_input[stim_on:stim_off, 4] = color_r * np.ones([stim_off - stim_on])
    color_input[stim_on:stim_off, 5] = color_l * np.ones([stim_off - stim_on])

    # Noise
    noise = np.sqrt(2 / alpha * sigma_in * sigma_in) * np.random.multivariate_normal(
        [0, 0, 0, 0, 0, 0], np.eye(n_in), n_t)

    # Baseline
    baseline = baseline * np.ones([n_t, n_in])

    # Input stream is rectified sum of baseline, task and noise signals.
    input_stream = np.maximum(baseline + cue_input + motion_input + color_input + noise, 0)

    # Target stream
    target_stream = 0.2 * np.ones([n_t, 2])
    if (context == "motion" and motion_coh > 0) or (context == "color" and color_coh > 0):
        target_stream[dec_on:dec_off, 0] = 1.2 * np.ones([dec_off - dec_on, 1]).squeeze()
        target_stream[dec_on:dec_off, 1] = 0.2 * np.ones([dec_off - dec_on, 1]).squeeze()
    else:
        target_stream[dec_on:dec_off, 0] = 0.2 * np.ones([dec_off - dec_on, 1]).squeeze()
        target_stream[dec_on:dec_off, 1] = 1.2 * np.ones([dec_off - dec_on, 1]).squeeze()

    return input_stream, target_stream

def generate_dataset_new(pfc_df, trial_events, alpha=.2, tau=200, sample_rate=5):

    """
    :param pfc_hidden:
    :param contexts:
    :param motion_cohs:
    :param color_cohs:
    :return: I_train, I_test, Y_train, Y_test
    """

    conditions = pfc_df[['context', 'stim_dir', 'stim_col2dir']].values

    # I_train = []
    # I_test = []
    # Y_train = []
    # Y_test = []
    I = []
    Y = []
    nun = pfc_df.response.values[0].shape[1]
    for condition in conditions:
        inputs, targets = generate_input_target_stream(condition[0],
                                                       condition[1],
                                                       condition[2],
                                                       alpha=alpha,
                                                       sigma_in=0.01,
                                                       baseline=0.2,
                                                       **trial_events)
        I.append(inputs)

        # Compute mean pfc response for this condition by sub-sampling trials
        pfc_df_cond = pfc_df[(pfc_df.context == condition[0]) & (pfc_df.stim_dir == condition[1]) & (
                    pfc_df.stim_col2dir == condition[2])]

        response = np.zeros((trial_events['n_t'], nun))
        response[trial_events['data_on']::sample_rate, :] = pfc_df_cond.response.values[0]
        Y.append(np.concatenate((response, targets), axis=1))

    I = torch.tensor(np.stack(I, axis=0)).float()
    Y = torch.tensor(np.stack(Y, axis=0)).float()

    return I, Y

In [51]:


u = []
Y = []
z = []
alpha = 0.2
for condition in flattened_conditions:
    inputs, targets = generate_input_target_stream(condition[0],
                                                   condition[1],
                                                   condition[2],
                                                   alpha=alpha,
                                                   sigma_in=0.01,
                                                   baseline=0.2,
                                                   **trial_events)
    u.append(inputs)
    z.append(targets)
    # Compute mean pfc response for this condition by sub-sampling trials
    # pfc_df_cond = pfc_df[(pfc_df.context == condition[0]) & (pfc_df.stim_dir == condition[1]) & (
    #             pfc_df.stim_col2dir == condition[2])]

    # response = np.zeros((trial_events['n_t'], nun))
    # response[trial_events['data_on']::sample_rate, :] = pfc_df_cond.response.values[0]
    # Y.append(np.concatenate((response, targets), axis=1))

u = torch.tensor(np.stack(u, axis=0)).float()
z = torch.tensor(np.stack(z, axis=0)).float()
#Y = torch.tensor(np.stack(Y, axis=0)).float()

In [54]:
z.shape, u.shape

(torch.Size([1800, 150, 2]), torch.Size([1800, 150, 6]))

## Fit latent net

In [57]:
from latent_net import *
latent_net = LatentNet(n=8, N=483 ,input_size=6, n_trials = u.shape[0],sigma_rec = 0.15)



# Fit latent circuit model
loss_history = latent_net.fit(u.detach(),z.detach(),y.detach(),epochs = 500,lr = .02,l_y = 1,weight_decay = 0.001)

/Users/cl1704/anaconda3/envs/testenv4/lib/python3.8/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([128, 15, 483])) that is different to the input size (torch.Size([128, 150, 483])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (150) must match the size of tensor b (15) at non-singleton dimension 1

In [58]:
u.shape, z.shape, y.shape

(torch.Size([1800, 150, 6]),
 torch.Size([1800, 150, 2]),
 torch.Size([1800, 15, 483]))